# img2txt: полный дерматоскопический пайплайн

Пайплайн из 4 шагов:
1. Извлечение признаков — сегментация + 60+ признаков
2. Бакетирование — числовые признаки -> категориальные метки
3. Ранжирование — XGBoost выбирает топ-10 важных признаков
4. Генерация текста — Qwen2.5-7B генерирует клиническое описание на русском

In [ ]:
import pandas as pd
import torch
from pathlib import Path
from extraction.feature_extraction_batch import extract_features_batch, images_to_df
from analysis.feature_bucketing_batch import bucket_features_batch
from importance.importance_inference import rank_features_batch
from generation.description_inference import generate_descriptions_batch
from generation.classification_types import ClassificationResult, Structure, FeatureType
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
# Пути
IMAGE_DIR = "/kaggle/input/datasets/mihailodin1/all-image-skin"
YOLO_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_yolo.pt"
UNET_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_unet.pth"
IMPORTANCE_CHECKPOINT = "training_v2/checkpoints/xgb_importance.pkl" # Обновлен под XGBoost!
FEATURES_CSV = "features_dataset.csv"

## Часть 1. Инференс для папки с изображениями (Batch Processing)

df = images_to_df(IMAGE_DIR).head(5)
df = extract_features_batch(df, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS)
df.to_csv(FEATURES_CSV, index=False)
print(f"Извлечено: {len(df)} изображений, успешно: {(df['status'] == 'success').sum()}")

## Шаг 2. Бакетирование


In [ ]:
df = bucket_features_batch(df)
print("Добавлены колонки: labels, features_organized, labels_json")

## Шаг 3. Ранжирование (Выбор топ-10 важных признаков XGBoost)

In [ ]:
df = rank_features_batch(df, importance_model_path=IMPORTANCE_CHECKPOINT, k=10)
print("Пример важных меток для первого изображения:")
print(df["important_labels"].iloc[0])

## Шаг 4. Генерация клинического описания (Qwen2.5-7B)

In [ ]:
df = generate_descriptions_batch(df, device=device)
print("\nПример сгенерированного описания:")
print(df["description"].iloc[0])
# Сохранение результатов
output_csv = "features_with_descriptions.csv"
df.to_csv(output_csv, index=False)
print(f"\nСохранено: {len(df)} строк в {output_csv}")

## Часть 2. Инференс для одного изображения (Single image)


In [ ]:
image_path = "/path/to/lesion.jpg"
if Path(image_path).exists():
    df_single = pd.DataFrame([{"image_path": image_path}])
    
    # 1. Извлечение
    df_single = extract_features_batch(
        df_single, 
        yolo_weights=YOLO_WEIGHTS, 
        unet_weights=UNET_WEIGHTS, 
        verbose=False
    )
    
    # 2. Бакетирование
    df_single = bucket_features_batch(df_single, verbose=False)
    
    # 3. Ранжирование (XGBoost)
    df_single = rank_features_batch(
        df_single, 
        importance_model_path=IMPORTANCE_CHECKPOINT, 
        verbose=False
    )
    
    # 4. Классификация и генерация описания
    # Можно вручную задать контекст (или получить из классификационной модели)
    df_single["classification"] = ClassificationResult(
        feature_type=FeatureType.SINGLE,
        structure=Structure.GLOBULES,
        properties=["однородный"],
        final_class="Меланома",
    )
    
    df_single = generate_descriptions_batch(
        df_single, 
        classification_col="classification", 
        device=device, 
        verbose=False
    )
    
    print("\nОписание для одного изображения:")
    print(df_single.iloc[0]["description"])
else:
    print(f"Файл {image_path} не найден. Укажите существующий путь (или загрузите фото) для теста.")